# 18BI · Week 2 exercise — first queries

**Business Intelligence** · FNSPE CTU in Prague · winter semester 2026/27

---

Last week you answered *"what was our revenue?"* by hand, from seven order lines, and
eight of you produced six different numbers. Today: the same question in SQL, on the same
seven rows.

It is light on purpose. Every query is a few lines. Under each exercise there are hints and
then a solution, all folded away — try first, open a hint when you are stuck, and the
solution only to check.

**Before you start:** [`00_hello_world.ipynb`](00_hello_world.ipynb) should print
`ALL GOOD`.

## 1 · Setup

DuckDB is an analytical database that lives inside Python — no server, no account. It can
read a CSV file as if it were a table.

The two lines below create **views**: a view is a saved query with a name. From now on
`orders` means *"whatever is in that CSV file"*, so you can write `from orders` instead of
the file path every time. This notebook lives in `exercises/`, so the data is one folder up.

In [ ]:
import duckdb

duckdb.sql("create or replace view orders  as from '../data/handout/orders.csv'")
duckdb.sql("create or replace view refunds as from '../data/handout/refunds.csv'")

`from orders` is DuckDB shorthand for `select * from orders`. Here is the handout, as a table:

In [ ]:
duckdb.sql("from orders")

In [ ]:
duckdb.sql("from refunds")

## 2 · Lines are not orders

**2.1** — How many rows does `orders` have? How many *different* orders? Why are the two
numbers not the same?

In [ ]:
# your query here

<details><summary>Hint</summary>

`count(*)` counts rows. `count(distinct order_id)` counts how many different values the
`order_id` column contains. Both fit in one `select`, separated by a comma.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select count(*) as lines, count(distinct order_id) as orders
from orders
""")
```

**7 lines, 6 orders.** Order O-1 has two lines. One row of this table is one *order line*,
not one order — this is the table's **grain**, and you will meet that word every week.

</details>

## 3 · Your number from last week

Last week you submitted a number and a rule (all eight responses are in
[lecture 01, §5](../lectures/01_bi_in_the_age_of_ai.ipynb)). Write **your** rule as a query.

Do you get your number back? If not — was it the rule, or the arithmetic?

In [ ]:
# your rule, as a query

<details><summary>Hint — a rule is five decisions</summary>

Start from the simplest possible query and add one decision at a time, running it after each:

```python
duckdb.sql("select sum(quantity * unit_price_net) as revenue from orders")
```

1. **Which orders count?** All of them, or only some: `where status = 'completed'`
2. **Discounts?** Subtract them per line: `quantity * unit_price_net - discount_net`
3. **Shipping?** Ignore it, add it, or subtract it as a cost — and if you add it, once per
   *order*, not once per line (section 5 shows why)
4. **VAT?** The prices are net. To report gross, multiply by `(1 + vat_rate)`
5. **The refund?** It lives in its own table: `- (select sum(refund_amount_net) from refunds)`

Every combination of these five answers is a different, defensible "revenue".

</details>

*No solution here: it is your rule. If your query and your submitted number disagree, trust
the query — it is the one you can check.*

## 4 · Three rules, three queries

Three of the six rules from the handout. Each one is a small change to the previous one.

**4.1** — Completed orders only, minus discounts. No shipping, no VAT, refund ignored.

In [ ]:
# your query here

<details><summary>Hint 1</summary>

Write the revenue of a single line first — `quantity * unit_price_net - discount_net` — and
look at it per row. Then wrap it in `sum(...)` to add all the lines up.

</details>

<details><summary>Hint 2</summary>

Keep only completed orders with `where status = 'completed'`.

The discount is a total for the whole line, so it is subtracted *after* multiplying. Last
week one rule wrote it as `(unit price − discount) × quantity`. On completed orders the two
happen to agree, because the only discounted completed line has a quantity of 1 — but run
both on all seven rows and you get **20 600** instead of **21 200**. Right by luck is not right.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select sum(quantity * unit_price_net - discount_net) as revenue
from orders
where status = 'completed'
""")
```

**15 500.** Rule C.

</details>

**4.2** — The same, but net of the refund. The refund lives in its own table.

In [ ]:
# your query here

<details><summary>Hint</summary>

A query in brackets that returns a single number — a **scalar subquery** — can be used
like any other number:

```sql
select (select ...) - (select ...) as revenue
```

Put your query from 4.1 in the first pair of brackets, and the total of
`refund_amount_net` from `refunds` in the second.

Why not join the two tables on `order_id`? Here it would happen to work, because the
refunded order O-4 has a single line. If it had two lines, the join would attach the refund
to both of them and subtract it twice — the same trap as in section 5.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select
    (select sum(quantity * unit_price_net - discount_net)
     from orders where status = 'completed')
  - (select sum(refund_amount_net) from refunds)
  as revenue
""")
```

**13 500.** Rule D — the answer two of you gave last week.

</details>

**4.3** — Back to 4.1, but with VAT added, as the finance team would report it.
`vat_rate` is a column.

In [ ]:
# your query here

<details><summary>Hint</summary>

Multiplying by `(1 + vat_rate)` adds 21%. Watch the brackets:
`quantity * unit_price_net - discount_net * (1 + vat_rate)` adds VAT to the *discount only*
and gives 15 395. The whole line revenue has to be in brackets before you multiply.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select sum((quantity * unit_price_net - discount_net) * (1 + vat_rate)) as revenue
from orders
where status = 'completed'
""")
```

**18 755.** Rule F. Note that the prices in this table are *net*: VAT is added on top,
never taken out.

</details>

## 5 · The fan-out

The most common bug in business intelligence, and the one behind 19 240 last week.
Shipping is charged **once per order**, but the table repeats it on every line of that order.

**5.1** — Add up the `shipping_net` column for completed orders. Then look at order O-1:
what went wrong?

In [ ]:
# your query here

<details><summary>Hint</summary>

For the total: `sum(shipping_net)` with the same `where` as in 4.1. To look at O-1 alone:

```python
duckdb.sql("select order_id, order_line, shipping_net from orders where order_id = 'O-1'")
```

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select sum(shipping_net) as shipping
from orders
where status = 'completed'
""")
```

**400 — and it is wrong.** O-1 has two lines, and both carry its 100 CZK delivery, so it
was counted twice. The real total is 300. Nothing errored, nothing warned you, and 400 looks
perfectly reasonable. That is what makes it dangerous.

</details>

**5.2** — Fix it. First get **one row per order**, then add up.

In [ ]:
# your query here

<details><summary>Hint 1</summary>

Write the inner query first and run it on its own — it should return four rows, one per
completed order:

```sql
select order_id, max(shipping_net) as shipping
from orders
where status = 'completed'
group by order_id
```

Why `max`? `group by` squeezes all lines of an order into one row, so every other column
needs an aggregate. All lines of an order carry the same shipping, so `max`, `min` or
`any_value` all pick the same number.

</details>

<details><summary>Hint 2</summary>

Now wrap it: `select sum(shipping) from ( ...the inner query... )`. A query in brackets
after `from` behaves like a table.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select sum(shipping) as shipping
from (
    select order_id, max(shipping_net) as shipping
    from orders
    where status = 'completed'
    group by order_id
)
""")
```

**300.** The shape of the fix is worth remembering: *change the grain first, then
aggregate*. The inner query turns order lines into orders; only then do we sum.

</details>

**5.3** *(optional)* — Put it together: rule E. Completed orders, minus discounts, **plus**
shipping — charged once per order.

In [ ]:
# your query here

<details><summary>Hint</summary>

You already have both halves: the query from 4.1 and the query from 5.2. Combine them with
the scalar-subquery trick from 4.2: `select (select ...) + (select ...)`.

</details>

<details><summary>Solution</summary>

```python
duckdb.sql("""
select
    (select sum(quantity * unit_price_net - discount_net)
     from orders where status = 'completed')
  + (select sum(shipping) from (
        select order_id, max(shipping_net) as shipping
        from orders where status = 'completed'
        group by order_id))
  as revenue
""")
```

**15 800.** Rule E. Sum the shipping column naively instead and you get 15 900 — the
fan-out, one more time.

</details>

## 6 · What SQL did not solve

SQL fixed the arithmetic. Every query above is exact, repeatable and runs in milliseconds,
and the slips that happened on paper last week cannot happen here.

**It did not choose the rule.** You still picked which query to run — and so would everyone
else in the company, each with their own notebook, each with a defensible answer.

Next week we put the rule in **one place**, so that nobody has to choose again.